# Data cleaning: Digikala product reviews and product catalog

Turns the raw `comments.parquet` and `products.parquet` into analysis-ready tables, following the findings of `eda_v2.ipynb`. The tables are meant for the course project (shopping assistant with retrieval over reviews, and prediction of `recommendation_status`).

| | |
|---|---|
| **Input** | `comments.parquet` (6.16M rows), `products.parquet` (1.28M rows), in the notebook folder |
| **Output** (`clean_output/`) | `comments_clean.parquet`, `products_clean.parquet`, `comments_with_products.parquet` (optional merged table), `cleaning_log.csv`, `validation_checks.csv`, `quality_report_comments.csv`, `quality_report_products.csv`, `key_overlap_by_id_digits.csv` |
| **Environment** | Python 3.12, pandas 3.x, numpy, pyarrow (same as the EDA) |
| **Runtime** | Text normalisation runs a Python function over every text and takes the longest (the log shows elapsed minutes per step). Set `SAMPLE_ROWS = 200_000` in the configuration to test the notebook quickly. |

## Design principles
1. **Every rule traces back to an EDA finding** (the table below). Nothing is removed "just in case".
2. **Prefer flags and missing values over deleting rows.** Rows are dropped only for exact duplicates, conflicting duplicate ids and comments without any body text.
3. **All choices are switches** in the configuration cell, so the next person can change a decision without rewriting code.
4. **Everything is logged** (`cleaning_log`: rows before, rows after, rows affected, elapsed minutes).
5. **The raw files are never modified.**
6. **Validation never wastes a long run:** checks are collected in a table, the files are saved, and only then does the last cell raise an error if a check failed.

## Pipeline

| Step | Rule | EDA evidence |
|---|---|---|
| 1 | `""`, `nan`, `None`, `[]`, `null` (any case) become real missing values in every text column of both tables | missing values stored as text; fake spike at length 3 |
| 2 | Comments: drop exact duplicates; for a repeated id keep the row with most likes. Products: drop exact duplicates; for a repeated id keep the row with a valid price, then the highest `Rate_cnt` | 1,318 / 3,229 duplicate comment rows / ids; 323,129 / 335,144 duplicate product rows / ids |
| 3 | Unicode normalisation of texts and labels (Arabic letters to Persian, digits to ASCII, zero-width and control characters, whitespace, diacritics, elongated letters, zero-width joiner placement) | `\r\n` in bodies, Persian digits in brands, spelling variants of categories |
| 4 | `advantages` / `disadvantages`: parse the stringified list, normalise items, drop placeholder and blank items, join with ` \| `, add `n_advantages` / `n_disadvantages` | pros/cons stored as Python-list strings |
| 5 | Jalali `created_at` becomes `jalali_year`, `jalali_month` and a Gregorian `created_date` | `pd.to_datetime` cannot parse Jalali dates |
| 6 | Comments: `rate` above 5 or equal to 0 becomes missing. Products: `Price == 0`, `min_price_last_month == 0` and `Rate` with `Rate_cnt == 0` become missing; add `price_is_extreme`, `brand_is_generic` | `rate` of 2500; zero means unknown |
| 7 | Comments: drop rows without `body`; add `body_len`, `body_persian_ratio`, `is_short_body`, `is_mostly_non_persian`, `body_dup_count`, `is_repeat_in_product` | 637 empty bodies; many identical short comments |
| 8 | Products: add `n_comments`; final placeholder pass over both tables | |
| 9 | Key diagnostics between the two tables and the optional merged table | teammates could not merge the tables |
| 10 | Validation table (no abort), save outputs, final strict check | |

## Changes in this version
- **Fixed a crash:** pros/cons lists can contain placeholder items such as `nan` or `None`, which survived parsing and failed the validation (`placeholder left in disadvantages`). Placeholder items are now dropped inside the parser, matching is case-insensitive, and a final placeholder pass runs before validation.
- **Validation no longer aborts before saving** (see principle 6).
- **Key diagnostics (step 9):** prints the dtype and digit length of `products.id` and `comments.product_id`, the overlap between the tables and the share of products with comments per id length. Both keys are cast to `int64` explicitly.
- **Merged table:** `comments_with_products.parquet` has every comment with its product's columns (prefix `product_`), so nobody has to merge by hand (switch `WRITE_MERGED`).
- **Duplicate checks on products:** asserts that ids are unique and that no full-row duplicates remain, and reports rows that are identical except for the id.
- **Leakage helpers for the classification task:** `body_dup_count` (how many comments share exactly the same text) and `is_repeat_in_product` (same text repeated within one product). Use them to deduplicate texts or to split by group, otherwise identical comments land in both train and test.
- `is_short_body` is computed explicitly on non-missing bodies, and elapsed time is logged.

## Assumptions to confirm with the data owner
- **`rate == 0` means "no rating"** (80.5% of such comments are *recommended*, while rates like 0.05 are 83.5% *not recommended*). Switch: `ZERO_RATE_IS_MISSING`.
- **For a repeated comment id the most-liked row is the freshest** (likes only grow).
- **Price unit is unknown**; prices are only ever compared with each other.
- **Product ids with conflicting rows** (they differ mostly in `Price`, `Seller`, `Rate_cnt`: the same product captured at different times) keep a single row; multi-category information is lost for about 750 ids.
- Short comments are *flagged*, not dropped, by default. Note that `MIN_BODY_CHARS = 3` flags only a handful of comments; many short comments (`عالی`, `خوب`) are informative and should stay.

## Not done on purpose
Not fixed: the category hierarchy, class imbalance, the 14.5% missing `recommendation_status` (rows without a label must be excluded when training the classifier), the maximum of 400 comments per product, comments of products with `Is_Fake == True`, stop-word removal and tokenisation. These belong to modelling, not cleaning.

## 0. Configuration and helpers

In [1]:
import ast
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

COMMENTS_PATH = Path("comments.parquet")
PRODUCTS_PATH = Path("products.parquet")
OUTPUT_DIR = Path("clean_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_ROWS = None              # e.g. 200_000 to test on the first rows of the comments table
PLACEHOLDERS = ["", "nan", "none", "null", "[]"]   # compared in lower case
LIST_SEPARATOR = " | "

DROP_EMPTY_BODY = True          # drop comments without any body text
DROP_SHORT_BODY = False         # also drop bodies shorter than MIN_BODY_CHARS
MIN_BODY_CHARS = 3
MIN_PERSIAN_RATIO = 0.5         # below this share of Persian letters a body is flagged
ZERO_RATE_IS_MISSING = True     # comment rate 0 means "no rating"
MAX_COMMENT_RATE = 5
EXTREME_PRICE_IQR_FACTOR = 3.0  # fences on log10(price) per sub_category
GENERIC_BRAND = "متفرقه"
WRITE_MERGED = True             # also write comments_with_products.parquet
STRICT_VALIDATION = True        # raise at the very end if a validation check failed

cleaning_log = []
T0 = time.time()


def log_step(step, table, rows_before, rows_after, affected, note=""):
    elapsed = round((time.time() - T0) / 60, 1)
    cleaning_log.append({
        "step": step, "table": table, "rows_before": rows_before, "rows_after": rows_after,
        "rows_affected": affected, "note": note, "elapsed_min": elapsed,
    })
    print(f"[{table}] {step}: affected {affected:,} | rows {rows_before:,} -> {rows_after:,} {note} ({elapsed} min)")

## 1. Load data

In [2]:
for path in (COMMENTS_PATH, PRODUCTS_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path.resolve()}")

df_comm = pd.read_parquet(COMMENTS_PATH)
df_prod = pd.read_parquet(PRODUCTS_PATH)
if SAMPLE_ROWS:
    df_comm = df_comm.head(SAMPLE_ROWS).copy()

print(f"Comments: {df_comm.shape[0]:,} rows x {df_comm.shape[1]} cols")
print(f"Products: {df_prod.shape[0]:,} rows x {df_prod.shape[1]} cols")
print(f"Key dtypes before casting: products.id = {df_prod['id'].dtype}, comments.product_id = {df_comm['product_id'].dtype}")

assert df_prod["id"].notna().all() and df_comm["product_id"].notna().all(), "missing keys"
df_prod["id"] = df_prod["id"].astype("int64")
df_comm["product_id"] = df_comm["product_id"].astype("int64")

Comments: 6,156,289 rows x 15 cols
Products: 1,283,496 rows x 12 cols
Key dtypes before casting: products.id = int64, comments.product_id = int64


## 2. Step 1: placeholder strings become missing values

**Why.** The export wrote missing values as the text `nan` (and empty lists as `[]`), so `isna()` saw nothing missing. **Rule.** In every string column of both tables, a value that equals `""`, `nan`, `none`, `null` or `[]` (any case, after stripping) becomes a real missing value. The same function runs again at the end of the pipeline (step 8) because later steps can create new placeholders.

In [3]:
def placeholders_to_nan(df, table, label="placeholders to missing"):
    counts = {}
    for col in df.columns:
        if not pd.api.types.is_string_dtype(df[col]):
            continue
        is_placeholder = df[col].str.strip().str.lower().isin(PLACEHOLDERS)
        n = int(is_placeholder.sum())
        if n:
            df[col] = df[col].mask(is_placeholder)
            counts[col] = n
    log_step(label, table, len(df), len(df), sum(counts.values()), f"{len(counts)} columns")
    return pd.Series(counts, dtype="int64", name="replaced").sort_values(ascending=False).to_frame()


display(placeholders_to_nan(df_comm, "comments"))
display(placeholders_to_nan(df_prod, "products"))

[comments] placeholders to missing: affected 21,623,457 | rows 6,156,289 -> 6,156,289 8 columns (0.3 min)


,replaced
true_to_size_rate,6065516
disadvantages,5744017
advantages,5448084
title,2866415
recommendation_status,894426
seller_title,302181
seller_code,302181
body,637


[products] placeholders to missing: affected 223 | rows 1,283,496 -> 1,283,496 1 columns (0.3 min)


,replaced
Seller,223


## 3. Step 2: duplicates

**Why.** EDA found 1,318 exact duplicate comments and 3,229 repeated comment ids; 323,129 exact duplicate products and 335,144 repeated product ids (about 12k of them conflicting). **Rules.**
- Exact duplicates are dropped in both tables.
- Comments with a repeated id: keep the row with the most likes (then dislikes). Assumption: likes only grow, so the most-liked copy is the latest capture.
- Products with a repeated id: keep the row with a valid price (`Price > 0`), then the highest `Rate_cnt`. The table printed below shows which columns differ between conflicting rows (mostly `Price`, `Seller`, `Rate_cnt`: the same product captured at different times). Only one row per id is kept.

In [4]:
n0 = len(df_comm)
df_comm = df_comm.drop_duplicates()
log_step("drop exact duplicates", "comments", n0, len(df_comm), n0 - len(df_comm))

n0 = len(df_comm)
df_comm = (
    df_comm.sort_values(["likes", "dislikes"], ascending=False)
    .drop_duplicates("id", keep="first")
    .sort_index()
)
log_step("repeated id: keep most-liked row", "comments", n0, len(df_comm), n0 - len(df_comm))

n0 = len(df_prod)
df_prod = df_prod.drop_duplicates()
log_step("drop exact duplicates", "products", n0, len(df_prod), n0 - len(df_prod))

conflicts = df_prod[df_prod["id"].duplicated(keep=False)]
if len(conflicts):
    differing = (conflicts.groupby("id").nunique(dropna=False) > 1).sum()
    print("Conflicting product ids: number of ids that differ per column")
    display(differing[differing > 0].sort_values(ascending=False).to_frame("ids_with_different_values"))

n0 = len(df_prod)
df_prod = (
    df_prod.assign(_invalid_price=df_prod["Price"].le(0))
    .sort_values(["_invalid_price", "Rate_cnt"], ascending=[True, False])
    .drop_duplicates("id", keep="first")
    .drop(columns="_invalid_price")
    .sort_values("id")
    .reset_index(drop=True)
)
log_step("repeated id: keep valid price, highest Rate_cnt", "products", n0, len(df_prod), n0 - len(df_prod))
df_comm = df_comm.reset_index(drop=True)

[comments] drop exact duplicates: affected 1,318 | rows 6,156,289 -> 6,154,971  (0.7 min)
[comments] repeated id: keep most-liked row: affected 1,911 | rows 6,154,971 -> 6,153,060  (1.4 min)
[products] drop exact duplicates: affected 323,129 | rows 1,283,496 -> 960,367  (1.5 min)
Conflicting product ids: number of ids that differ per column


,ids_with_different_values
Price,8254
Seller,4036
Rate_cnt,3251
min_price_last_month,1458
sub_category,757
Rate,646
title_fa,6
Brand,2


[products] repeated id: keep valid price, highest Rate_cnt: affected 12,015 | rows 960,367 -> 948,352  (1.5 min)


## 4. Step 3: text normalisation

**Why.** Texts contain `\r\n`, Arabic letter variants, Persian digits and inconsistent zero-width joiners; category and brand labels differ only by these details (for example `برس‌ها` vs `برس ها`).

**Rules** (function `normalize_one`, applied to every non-missing value):
1. Arabic `ي ك ى` become Persian `ی ک ی`; tatweel and Arabic diacritics are removed.
2. Persian and Arabic-Indic digits become ASCII digits.
3. Zero-width and bidi control characters (except the zero-width non-joiner) and non-breaking spaces are removed or turned into spaces.
4. All whitespace (including newlines and tabs) collapses to a single space; the text is stripped.
5. A run of three or more identical non-digit characters is shortened to two (`عالیییی` becomes `عالیی`). Digits are excluded so `1000` is untouched.
6. Repeated zero-width joiners collapse to one, and joiners next to a space or at the ends are removed.
7. For **labels** (brand, categories, seller, sub-category) the joiner is also replaced by a space so spelling variants merge.

Texts that become empty are set to missing. The table shows how many values each column changed.

In [5]:
ZERO_WIDTH = "\u200b\u200d\u200e\u200f\u202a\u202b\u202c\u202d\u202e\u2066\u2067\u2068\u2069\ufeff"
CHAR_MAP = str.maketrans({
    **{c: str(i) for i, c in enumerate("۰۱۲۳۴۵۶۷۸۹")},
    **{c: str(i) for i, c in enumerate("٠١٢٣٤٥٦٧٨٩")},
    "ي": "ی", "ى": "ی", "ك": "ک", "ـ": "", "\u00a0": " ",
    **{c: "" for c in ZERO_WIDTH},
})
ZWNJ = "\u200c"
RE_DIACRITICS = re.compile("[\u064b-\u065f\u0670]")
RE_SPACES = re.compile(r"\s+")
RE_REPEAT = re.compile(r"([^\d\s])\1{2,}")
RE_ZWNJ_RUN = re.compile(ZWNJ + "+")
RE_ZWNJ_EDGE = re.compile("^" + ZWNJ + "|" + ZWNJ + "$|(?<=\\s)" + ZWNJ + "|" + ZWNJ + "(?=\\s)")


def normalize_one(text, keep_zwnj=True):
    text = text.translate(CHAR_MAP)
    text = RE_DIACRITICS.sub("", text)
    text = RE_SPACES.sub(" ", text)
    text = RE_REPEAT.sub(r"\1\1", text)
    text = RE_ZWNJ_RUN.sub(ZWNJ, text)
    text = RE_ZWNJ_EDGE.sub("", text)
    if not keep_zwnj:
        text = text.replace(ZWNJ, " ")
    return RE_SPACES.sub(" ", text).strip()


def normalize_series(series, keep_zwnj=True):
    cleaned = series.map(lambda t: normalize_one(t, keep_zwnj), na_action="ignore")
    return cleaned.mask(cleaned.eq(""))


assert normalize_one("عاااالی\r\n  ۱۲۳  1000") == "عاالی 123 1000"
assert normalize_one("برس\u200cها", keep_zwnj=False) == "برس ها"
assert normalize_one("كتاب\u200c ي") == "کتاب ی"

changed = {}
for df, table, cols, keep in [
    (df_comm, "comments", ["title", "body"], True),
    (df_comm, "comments", ["seller_title"], False),
    (df_prod, "products", ["title_fa"], True),
    (df_prod, "products", ["Brand", "Category1", "Category2", "Seller", "sub_category"], False),
]:
    for col in cols:
        original = df[col]
        df[col] = normalize_series(original, keep_zwnj=keep)
        changed[(table, col)] = int((df[col].fillna("") != original.fillna("")).sum())
        log_step(f"normalize text: {col}", table, len(df), len(df), changed[(table, col)])

display(pd.Series(changed, name="values_changed").to_frame())

[comments] normalize text: title: affected 436,059 | rows 6,153,060 -> 6,153,060  (1.8 min)
[comments] normalize text: body: affected 1,833,685 | rows 6,153,060 -> 6,153,060  (3.5 min)
[comments] normalize text: seller_title: affected 541,115 | rows 6,153,060 -> 6,153,060  (4.0 min)
[products] normalize text: title_fa: affected 150,067 | rows 948,352 -> 948,352  (4.3 min)
[products] normalize text: Brand: affected 14,031 | rows 948,352 -> 948,352  (4.3 min)
[products] normalize text: Category1: affected 39,842 | rows 948,352 -> 948,352  (4.4 min)
[products] normalize text: Category2: affected 7,549 | rows 948,352 -> 948,352  (4.5 min)
[products] normalize text: Seller: affected 36,511 | rows 948,352 -> 948,352  (4.6 min)
[products] normalize text: sub_category: affected 0 | rows 948,352 -> 948,352  (4.7 min)


values_changed
comments title                 436059
         body                 1833685
         seller_title          541115
products title_fa              150067
         Brand                  14031
         Category1              39842
         Category2               7549
         Seller                 36511
         sub_category               0

## 5. Step 4: advantages and disadvantages

**Why.** Both columns are Python lists written as strings (`['a', 'b']`). **Rule.** Parse each string, normalise every item, drop blank and placeholder items (`nan`, `none`, `null`, `[]`) and join the rest with ` | `. Add `n_advantages` / `n_disadvantages` (0 when the field is empty). Lists without any valid item become missing. If a string cannot be parsed it is kept as a single item. (Placeholder items inside lists were the cause of a crash in the first version of this notebook.)

In [6]:
def parse_list_cell(text):
    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError, MemoryError, RecursionError):
        parsed = [text]
    items = parsed if isinstance(parsed, (list, tuple)) else [parsed]
    items = [normalize_one(str(item)) for item in items]
    return [item for item in items if item and item.lower() not in PLACEHOLDERS]


assert parse_list_cell("['nan', 'good', ' None ', '', '[]']") == ["good"]
assert parse_list_cell("['a', 'b']") == ["a", "b"]

for col in ["advantages", "disadvantages"]:
    lists = df_comm[col].dropna().map(parse_list_cell)
    counts = lists.map(len)
    joined = lists.map(LIST_SEPARATOR.join).mask(counts.eq(0))
    df_comm[f"n_{col}"] = counts.reindex(df_comm.index, fill_value=0).astype("int16")
    df_comm[col] = joined.reindex(df_comm.index)
    log_step(f"parse list: {col}", "comments", len(df_comm), len(df_comm), int(len(lists)),
             f"{int(counts.sum()):,} items, {int(counts.eq(0).sum()):,} lists without a valid item")

[comments] parse list: advantages: affected 707,811 | rows 6,153,060 -> 6,153,060 1,340,386 items, 112 lists without a valid item (5.0 min)
[comments] parse list: disadvantages: affected 412,040 | rows 6,153,060 -> 6,153,060 533,076 items, 168 lists without a valid item (5.1 min)


## 6. Step 5: Jalali dates

**Why.** `created_at` is a Jalali date string such as `23 شهریور 1402`. **Rule.** Extract day, month name and year (digits normalised), map the month name to 1-12, and convert to a Gregorian date with the standard 33-year-cycle arithmetic (no external package). New columns: `jalali_year`, `jalali_month`, `created_date`. Dates that do not match the pattern or that have an impossible day become missing. Unit tests with known Nowruz dates run before the conversion. The original `created_at` is not kept in the clean table (it stays in the raw file).

In [7]:
JALALI_MONTHS = {
    "فروردین": 1, "اردیبهشت": 2, "خرداد": 3, "تیر": 4, "مرداد": 5, "شهریور": 6,
    "مهر": 7, "آبان": 8, "آذر": 9, "دی": 10, "بهمن": 11, "اسفند": 12,
}
MONTH_START_DAY = np.array([0, 31, 62, 93, 124, 155, 186, 216, 246, 276, 306, 336])
MONTH_LENGTH = np.array([31] * 6 + [30] * 6)


def jalali_to_datetime64(year, month, day):
    jy = year - 979
    day_no = 365 * jy + (jy // 33) * 8 + ((jy % 33) + 3) // 4
    day_no = day_no + MONTH_START_DAY[month - 1] + (day - 1)
    return np.datetime64("1600-01-01") + (day_no + 79).astype("timedelta64[D]")


known = {
    (1395, 1, 1): "2016-03-20", (1398, 1, 1): "2019-03-21", (1399, 1, 1): "2020-03-20",
    (1400, 1, 1): "2021-03-21", (1401, 1, 1): "2022-03-21", (1402, 1, 1): "2023-03-21",
    (1403, 1, 1): "2024-03-20", (1402, 6, 23): "2023-09-14", (1401, 12, 29): "2023-03-20",
}
for (y, m, d), expected in known.items():
    got = jalali_to_datetime64(np.array([y]), np.array([m]), np.array([d]))[0]
    assert got == np.datetime64(expected), f"{(y, m, d)}: expected {expected}, got {got}"
print("Jalali conversion tests passed")

raw_dates = df_comm["created_at"].str.translate(CHAR_MAP)
parts = raw_dates.str.extract(r"(?P<day>\d{1,2})\s+(?P<month_name>\S+)\s+(?P<year>\d{4})")
year = pd.to_numeric(parts["year"], errors="coerce")
day = pd.to_numeric(parts["day"], errors="coerce")
month = parts["month_name"].map(JALALI_MONTHS)

month_length = pd.Series(MONTH_LENGTH[month.fillna(1).astype("int64").to_numpy() - 1], index=df_comm.index)
valid = year.notna() & month.notna() & day.notna() & day.ge(1) & day.le(month_length)

created = pd.Series(np.full(len(df_comm), np.datetime64("NaT"), dtype="datetime64[ns]"), index=df_comm.index)
created.loc[valid] = jalali_to_datetime64(
    year[valid].to_numpy("int64"), month[valid].to_numpy("int64"), day[valid].to_numpy("int64")
).astype("datetime64[ns]")

df_comm["created_date"] = created
df_comm["jalali_year"] = year.where(valid).astype("Int16")
df_comm["jalali_month"] = month.where(valid).astype("Int8")
log_step("parse Jalali dates", "comments", len(df_comm), len(df_comm), int((~valid).sum()), "rows left without a date")
print(f"Date range: {df_comm['created_date'].min().date()} to {df_comm['created_date'].max().date()}")

Jalali conversion tests passed
[comments] parse Jalali dates: affected 0 | rows 6,153,060 -> 6,153,060 rows left without a date (5.6 min)
Date range: 2016-07-13 to 2023-10-18


## 7. Step 6: numeric values

**Comments.** `rate` above 5 is impossible (EDA found 2500) and becomes missing. `rate == 0` is treated as "no rating" (switch `ZERO_RATE_IS_MISSING`; evidence in the EDA, section 5). The remaining values are in (0, 5] and keep their fractions.

**Products.**
- `Price == 0` and `min_price_last_month == 0` (94.2% of products) mean "unknown" and become missing.
- `Rate` is set to missing when `Rate_cnt == 0` (an unrated product has `Rate` 0, not a real score).
- `price_is_extreme`: `log10(Price)` outside the fences `Q1 - k*IQR` and `Q3 + k*IQR` computed per `sub_category` (k = `EXTREME_PRICE_IQR_FACTOR`). Only flagged, never removed.
- `brand_is_generic`: brand equals `متفرقه` ("miscellaneous", 55.9% of products).

In [8]:
rate = df_comm["rate"]
invalid_rate = rate.lt(0) | rate.gt(MAX_COMMENT_RATE)
zero_rate = rate.eq(0) if ZERO_RATE_IS_MISSING else pd.Series(False, index=rate.index)
df_comm["rate"] = rate.mask(invalid_rate | zero_rate)
log_step("comment rate to missing", "comments", len(df_comm), len(df_comm), int((invalid_rate | zero_rate).sum()),
         f"{int(invalid_rate.sum()):,} out of range, {int(zero_rate.sum()):,} zeros")

for col in ["Price", "min_price_last_month"]:
    zeros = int(df_prod[col].le(0).sum())
    df_prod[col] = df_prod[col].where(df_prod[col] > 0).astype("float64")
    log_step(f"{col} 0 to missing", "products", len(df_prod), len(df_prod), zeros)

unrated = df_prod["Rate_cnt"].eq(0)
df_prod["Rate"] = df_prod["Rate"].where(~unrated).astype("float64")
log_step("Rate without ratings to missing", "products", len(df_prod), len(df_prod), int(unrated.sum()))

log_price = np.log10(df_prod["Price"])
by_sub = log_price.groupby(df_prod["sub_category"])
q1 = df_prod["sub_category"].map(by_sub.quantile(0.25))
q3 = df_prod["sub_category"].map(by_sub.quantile(0.75))
iqr = q3 - q1
df_prod["price_is_extreme"] = (
    (log_price < q1 - EXTREME_PRICE_IQR_FACTOR * iqr) | (log_price > q3 + EXTREME_PRICE_IQR_FACTOR * iqr)
).fillna(False).astype(bool)
df_prod["brand_is_generic"] = df_prod["Brand"].eq(GENERIC_BRAND).fillna(False).astype(bool)
log_step("flag extreme prices", "products", len(df_prod), len(df_prod), int(df_prod["price_is_extreme"].sum()))
log_step("flag generic brand", "products", len(df_prod), len(df_prod), int(df_prod["brand_is_generic"].sum()))

[comments] comment rate to missing: affected 529,022 | rows 6,153,060 -> 6,153,060 1 out of range, 529,021 zeros (5.6 min)
[products] Price 0 to missing: affected 156 | rows 948,352 -> 948,352  (5.6 min)
[products] min_price_last_month 0 to missing: affected 893,172 | rows 948,352 -> 948,352  (5.6 min)
[products] Rate without ratings to missing: affected 587,363 | rows 948,352 -> 948,352  (5.6 min)
[products] flag extreme prices: affected 3,501 | rows 948,352 -> 948,352  (5.7 min)
[products] flag generic brand: affected 530,051 | rows 948,352 -> 948,352  (5.7 min)


## 8. Step 7: comment text quality

**Rules.**
- Comments without `body` (EDA: 637, plus bodies that became empty after normalisation) are dropped (`DROP_EMPTY_BODY`). Without a body there is no main text.
- `body_len`: number of characters. `is_short_body`: a non-missing body with fewer than `MIN_BODY_CHARS` characters (flagged by default; dropped only if `DROP_SHORT_BODY = True`). The default of 3 flags only a handful of comments on purpose: very short comments such as `عالی` carry a clear opinion.
- `body_persian_ratio`: share of characters in the Arabic/Persian Unicode block; `is_mostly_non_persian` flags values below `MIN_PERSIAN_RATIO`.
- **Leakage helpers.** `body_dup_count`: number of comments whose normalised body is exactly the same (a comment such as `خوب` appears thousands of times). `is_repeat_in_product`: the same body already appeared earlier for the same product. If identical texts end up in both the training and the test set, the classifier's score is inflated; deduplicate on `body` or split by group.

In [9]:
body = df_comm["body"]
body_len = body.str.len()
has_body = body.notna()
df_comm["body_len"] = body_len.astype("Int32")
df_comm["body_persian_ratio"] = (body.str.count("[\u0600-\u06FF]") / body_len).round(3).astype("float32")
df_comm["is_short_body"] = (has_body & body_len.lt(MIN_BODY_CHARS)).astype(bool)
df_comm["is_mostly_non_persian"] = (has_body & df_comm["body_persian_ratio"].lt(MIN_PERSIAN_RATIO)).astype(bool)

print(f"Bodies missing: {int((~has_body).sum()):,}")
for flag in ["is_short_body", "is_mostly_non_persian"]:
    log_step(f"flag {flag}", "comments", len(df_comm), len(df_comm), int(df_comm[flag].sum()))

n0 = len(df_comm)
if DROP_EMPTY_BODY:
    df_comm = df_comm[df_comm["body"].notna()]
    log_step("drop comments without body", "comments", n0, len(df_comm), n0 - len(df_comm))

n0 = len(df_comm)
if DROP_SHORT_BODY:
    df_comm = df_comm[~df_comm["is_short_body"]]
    log_step("drop short bodies", "comments", n0, len(df_comm), n0 - len(df_comm))

df_comm = df_comm.reset_index(drop=True)
df_comm["body_dup_count"] = df_comm.groupby("body")["body"].transform("size").fillna(0).astype("int32")
df_comm["is_repeat_in_product"] = df_comm.duplicated(["product_id", "body"], keep="first")
log_step("flag is_repeat_in_product", "comments", len(df_comm), len(df_comm), int(df_comm["is_repeat_in_product"].sum()))
print(f"Comments whose exact text appears more than once: {(df_comm['body_dup_count'] > 1).mean():.1%}")

Bodies missing: 852
[comments] flag is_short_body: affected 852 | rows 6,153,060 -> 6,153,060  (6.0 min)
[comments] flag is_mostly_non_persian: affected 5,339 | rows 6,153,060 -> 6,153,060  (6.0 min)
[comments] drop comments without body: affected 852 | rows 6,153,060 -> 6,152,208  (6.2 min)
[comments] flag is_repeat_in_product: affected 254,995 | rows 6,152,208 -> 6,152,208  (6.4 min)
Comments whose exact text appears more than once: 37.9%


## 9. Step 8: links between the tables and final placeholder pass

**Rules.** `n_comments` is the number of cleaned comments per product (0 for products without comments). The placeholder function runs once more over both tables, because normalisation and list parsing can create new placeholders.

In [10]:
df_prod["n_comments"] = df_prod["id"].map(df_comm["product_id"].value_counts()).fillna(0).astype("int32")

display(placeholders_to_nan(df_comm, "comments", "final placeholder pass"))
display(placeholders_to_nan(df_prod, "products", "final placeholder pass"))

[comments] final placeholder pass: affected 0 | rows 6,152,208 -> 6,152,208 0 columns (6.6 min)


,replaced


[products] final placeholder pass: affected 0 | rows 948,352 -> 948,352 0 columns (6.6 min)


,replaced


## 10. Step 9: key diagnostics between the two tables

A teammate reported that `products.id` and `comments.product_id` did not seem to match (6-digit vs 7-digit ids) and that only about 0.1% of products had comments. This step prints everything needed to check that on the cleaned data: key dtypes, id length, how many comments find their product, and the share of products that have comments **per id length**. Note that most products legitimately have **no** comments (the EDA found about 35%), so a merge must start from the comments table (left join) and not from the products table. If the overlap printed here is much lower than expected, the merge was probably done on a sample of one table only, on mismatched dtypes or on the wrong columns.

In [11]:
def digit_count(ids):
    return (np.floor(np.log10(ids.clip(lower=1))) + 1).astype("int64")


print(f"dtypes: products.id = {df_prod['id'].dtype}, comments.product_id = {df_comm['product_id'].dtype}")
print(f"products.id range: {df_prod['id'].min():,} to {df_prod['id'].max():,}")
print(f"comments.product_id range: {df_comm['product_id'].min():,} to {df_comm['product_id'].max():,}")

comments_found = df_comm["product_id"].isin(df_prod["id"])
print(f"Comments whose product exists in the catalog: {comments_found.mean():.2%} ({int(comments_found.sum()):,} of {len(df_comm):,})")
print(f"Products with at least one comment: {(df_prod['n_comments'] > 0).sum():,} of {len(df_prod):,} "
      f"({(df_prod['n_comments'] > 0).mean():.1%})")

key_overlap = (
    pd.DataFrame({"id_digits": digit_count(df_prod["id"]), "has_comments": df_prod["n_comments"] > 0})
    .groupby("id_digits")["has_comments"].agg(products="size", with_comments="sum")
)
key_overlap["share_with_comments_percent"] = (key_overlap["with_comments"] / key_overlap["products"] * 100).round(2)
key_overlap["comments"] = digit_count(df_comm["product_id"]).value_counts().reindex(key_overlap.index, fill_value=0)
key_overlap.to_csv(OUTPUT_DIR / "key_overlap_by_id_digits.csv", encoding="utf-8-sig")
print("Products and comments by number of digits of the product id")
display(key_overlap)

print(f"Product rows identical except for the id: {int(df_prod.drop(columns='id').duplicated().sum()):,}")

dtypes: products.id = int64, comments.product_id = int64
products.id range: 7,602 to 12,773,763
comments.product_id range: 7,602 to 12,702,212
Comments whose product exists in the catalog: 100.00% (6,152,208 of 6,152,208)
Products with at least one comment: 331,586 of 948,352 (35.0%)
Products and comments by number of digits of the product id


,products,with_comments,share_with_comments_percent,comments
id_digits,,,,
4,1,1,100.00,200
5,5820,4549,78.16,188136
6,36245,25416,70.12,1025326
7,565892,249258,44.05,4693939
8,340394,52362,15.38,244607


Product rows identical except for the id: 16,146


## 11. Step 10: validation (collected, not aborting)

Every check is stored in a table. The notebook keeps going and saves the files; the very last cell raises an error if any check failed (`STRICT_VALIDATION`).

In [12]:
def no_placeholders(series):
    return not series.dropna().str.strip().str.lower().isin(PLACEHOLDERS).any()


checks = {
    "comment ids unique": df_comm["id"].is_unique,
    "product ids unique": df_prod["id"].is_unique,
    "no full-row duplicates in products": not df_prod.duplicated().any(),
    "no full-row duplicates in comments": not df_comm.duplicated().any(),
    "all comments have a product in the catalog": bool(df_comm["product_id"].isin(df_prod["id"]).all()),
    "recommendation_status has only the 3 expected classes": set(df_comm["recommendation_status"].dropna().unique())
        <= {"recommended", "not_recommended", "no_idea"},
    "rate within (0, MAX] or missing": bool(df_comm["rate"].dropna().between(0, MAX_COMMENT_RATE).all())
        and (not ZERO_RATE_IS_MISSING or bool((df_comm["rate"].dropna() > 0).all())),
    "prices positive or missing": bool((df_prod["Price"].dropna() > 0).all())
        and bool((df_prod["min_price_last_month"].dropna() > 0).all()),
    "Rate missing when Rate_cnt == 0": bool(df_prod.loc[df_prod["Rate_cnt"] == 0, "Rate"].isna().all()),
    "is_short_body only on non-missing bodies": not (df_comm["is_short_body"] & df_comm["body"].isna()).any(),
    "bodies present (if DROP_EMPTY_BODY)": (not DROP_EMPTY_BODY) or bool(df_comm["body"].notna().all()),
    "at least 95% of dates parsed": df_comm["created_date"].notna().mean() > 0.95,
}
for col in ["title", "body", "advantages", "disadvantages", "recommendation_status", "seller_title", "seller_code", "true_to_size_rate"]:
    checks[f"no placeholder left in comments.{col}"] = no_placeholders(df_comm[col])
for col in ["Brand", "Category1", "Category2", "Seller", "title_fa"]:
    checks[f"no placeholder left in products.{col}"] = no_placeholders(df_prod[col])

check_table = pd.Series(checks, name="passed").astype(bool).to_frame()
check_table.to_csv(OUTPUT_DIR / "validation_checks.csv", encoding="utf-8-sig")
failed_checks = check_table.index[~check_table["passed"]].tolist()
display(check_table)
print("All validation checks passed" if not failed_checks else f"FAILED checks: {failed_checks}")

,passed
comment ids unique,True
product ids unique,True
no full-row duplicates in products,True
no full-row duplicates in comments,True
all comments have a product in the catalog,True
recommendation_status has only the 3 expected classes,True
"rate within (0, MAX] or missing",True
prices positive or missing,True
Rate missing when Rate_cnt == 0,True
is_short_body only on non-missing bodies,True


All validation checks passed


## 12. Save outputs

Writes the two clean tables, the optional merged table (every comment with its product's columns, prefixed `product_`; `product_id` is the join key), the step log and the quality reports. The column order below is the output schema documented in `README.md`.

In [13]:
COMMENT_COLUMNS = [
    "id", "product_id", "created_date", "jalali_year", "jalali_month",
    "title", "body", "body_len", "body_persian_ratio", "is_short_body", "is_mostly_non_persian",
    "body_dup_count", "is_repeat_in_product",
    "advantages", "n_advantages", "disadvantages", "n_disadvantages",
    "rate", "recommendation_status", "is_buyer", "likes", "dislikes",
    "seller_title", "seller_code", "true_to_size_rate",
]
PRODUCT_COLUMNS = [
    "id", "title_fa", "Category1", "Category2", "sub_category", "Brand", "brand_is_generic", "Seller",
    "Price", "price_is_extreme", "min_price_last_month", "Rate", "Rate_cnt", "Is_Fake", "n_comments",
]

comments_clean = df_comm[COMMENT_COLUMNS].sort_values("id").reset_index(drop=True)
products_clean = df_prod[PRODUCT_COLUMNS].sort_values("id").reset_index(drop=True)

comments_clean.to_parquet(OUTPUT_DIR / "comments_clean.parquet", index=False)
products_clean.to_parquet(OUTPUT_DIR / "products_clean.parquet", index=False)

if WRITE_MERGED:
    products_prefixed = products_clean.rename(columns={c: f"product_{c}" for c in products_clean.columns})
    merged = comments_clean.merge(products_prefixed, on="product_id", how="left", validate="many_to_one")
    assert len(merged) == len(comments_clean), "merge changed the number of comments"
    merged.to_parquet(OUTPUT_DIR / "comments_with_products.parquet", index=False)
    print(f"comments_with_products: {merged.shape[0]:,} rows x {merged.shape[1]} cols")
    del merged

log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")


def quality_report(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
    })


for name, table in [("comments", comments_clean), ("products", products_clean)]:
    quality_report(table).to_csv(OUTPUT_DIR / f"quality_report_{name}.csv", encoding="utf-8-sig")

print(f"comments_clean: {comments_clean.shape[0]:,} rows x {comments_clean.shape[1]} cols")
print(f"products_clean: {products_clean.shape[0]:,} rows x {products_clean.shape[1]} cols")
print(f"Saved to {OUTPUT_DIR.resolve()}")

comments_with_products: 6,152,208 rows x 39 cols
comments_clean: 6,152,208 rows x 25 cols
products_clean: 948,352 rows x 15 cols
Saved to C:\Users\Ramtin\Desktop\LLM\clean_output


## 13. Result: cleaning log and quality reports

In [14]:
print("Cleaning log")
display(log_df)
print("Comments: quality report")
display(quality_report(comments_clean))
print("Products: quality report")
display(quality_report(products_clean))
display(comments_clean.head(3))
display(products_clean.head(3))

Cleaning log


,step,table,rows_before,rows_after,rows_affected,note,elapsed_min
0,placeholders to missing,comments,6156289,6156289,21623457,8 columns,0.3
1,placeholders to missing,products,1283496,1283496,223,1 columns,0.3
2,drop exact duplicates,comments,6156289,6154971,1318,,0.7
3,repeated id: keep most-liked row,comments,6154971,6153060,1911,,1.4
4,drop exact duplicates,products,1283496,960367,323129,,1.5
5,"repeated id: keep valid price, highest Rate_cnt",products,960367,948352,12015,,1.5
6,normalize text: title,comments,6153060,6153060,436059,,1.8
7,normalize text: body,comments,6153060,6153060,1833685,,3.5
8,normalize text: seller_title,comments,6153060,6153060,541115,,4.0
9,normalize text: title_fa,products,948352,948352,150067,,4.3


Comments: quality report


,dtype,missing_count,missing_percent
id,int64,0,0.00
product_id,int64,0,0.00
created_date,datetime64[ns],0,0.00
jalali_year,Int16,0,0.00
jalali_month,Int8,0,0.00
title,str,2866712,46.60
body,str,0,0.00
body_len,Int32,0,0.00
body_persian_ratio,float32,0,0.00
is_short_body,bool,0,0.00


Products: quality report


,dtype,missing_count,missing_percent
id,int64,0,0.00
title_fa,str,0,0.00
Category1,str,0,0.00
Category2,str,181346,19.12
sub_category,str,0,0.00
Brand,str,0,0.00
brand_is_generic,bool,0,0.00
Seller,str,156,0.02
Price,float64,156,0.02
price_is_extreme,bool,0,0.00


,id,product_id,created_date,jalali_year,jalali_month,title,body,body_len,body_persian_ratio,is_short_body,is_mostly_non_persian,body_dup_count,is_repeat_in_product,advantages,n_advantages,disadvantages,n_disadvantages,rate,recommendation_status,is_buyer,likes,dislikes,seller_title,seller_code,true_to_size_rate
0,981586,56962,2016-07-20,1395,4,لطفا موجودش کنید.,بوی خوبی با توجه به قیمت و همه رده هاش داره ام...,161,0.764,False,False,1,False,قیمت مناسب,1,با توجه به قیمت، هیچی!,1,3.4,NaN,False,7,1,NaN,NaN,NaN
1,989879,62058,2016-07-13,1395,4,عطری ملایم,در ابتدا وقتی استفاده می کنید عطری بسیار ملایم...,194,0.784,False,False,1,False,NaN,0,NaN,0,2.9,NaN,False,7,0,NaN,NaN,NaN
2,989923,75164,2016-07-14,1395,4,واقعا عالیه,رایحش جوریه که هر سلیقه ای می پسنده. بخرید پشی...,61,0.787,False,False,1,False,NaN,0,NaN,0,5.0,NaN,False,6,0,NaN,NaN,NaN


,id,title_fa,Category1,Category2,sub_category,Brand,brand_is_generic,Seller,Price,price_is_extreme,min_price_last_month,Rate,Rate_cnt,Is_Fake,n_comments
0,7602,دسته بازی دوال شاک مخصوص پلی استیشن 3,لوازم جانبی کنسول بازی,NaN,toys and kids,سونی,False,رسپان مارکت,3290000.0,False,NaN,74.0,274,False,200
1,12298,ساعت مچی عقربه ای مردانه کاسیو جی شاک GA-110-1ADR,اکسسوری مردانه,ساعت مردانه,clothe,کاسیو,False,دیجی کالا,57312000.0,False,57133000.0,64.0,21,False,0
2,12302,ساعت مچی دیجیتالی مردانه کاسیو جی شاک GD-100-1BDR,اکسسوری مردانه,ساعت مردانه,clothe,کاسیو,False,دیجی کالا,44154000.0,False,43063000.0,90.0,8,False,9


## 14. Final check

Raises an error if any validation check failed. The files have already been written at this point, so a failure never costs a re-run; look at `validation_checks.csv`, fix the rule and re-run.

In [15]:
if STRICT_VALIDATION and failed_checks:
    raise AssertionError(f"Validation failed: {failed_checks}")
print("Cleaning finished successfully")

Cleaning finished successfully


## How to extend or change this notebook
- To change a decision, edit the switch in section 0 and re-run from the top; the log shows the effect.
- To add a rule, add a function, a `log_step(...)` call and a check in the validation dictionary.
- The clean files are the input for modelling. Re-running reads the raw files again, so the raw data are never overwritten.
- Join the tables on `comments_clean.product_id == products_clean.id` (a left join from comments), or use `comments_with_products.parquet`.